# Categorical Feature Experiment: room_type / property_type / neighbourhood_cleansed

Standalone experiment -- does **not** touch `main.py` / `notebooks/pipeline.ipynb`.

Right now the main pipeline only feeds `select_dtypes(include=[np.number])` into
the models, which silently drops every categorical column: `room_type`,
`property_type`, and `neighbourhood_cleansed` are never seen by the model, only
raw `latitude`/`longitude`. This notebook tests whether encoding those
categoricals actually improves XGBoost, using the **same train/test split** for
both variants so the comparison is apples-to-apples.

- **Baseline**: current numeric-only feature set (lat/lon, accommodates, etc.)
- **+ categoricals**: baseline + one-hot `room_type`, top-15 one-hot `property_type`
  (rest bucketed as "Other"), and frequency-encoded `neighbourhood_cleansed`

Encoders are fit on the training split only, to avoid leakage.


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import CLEANED_TARGET_PATH, RANDOM_STATE, RAW_LISTINGS_PATH, TARGET_COLUMN, TEST_SIZE
from src.data.clean import clean_missing_target, clean_numerical_features, clean_price_column, remove_price_outliers_dynamic
from src.evaluate import print_metrics_log_target
from src.features.categorical import fit_categorical_encoders, transform_categorical_features
from src.features.selection import select_important_features
from src.features.text import add_amenity_count, add_description_word_count
from src.models.tree_models import train_xgboost


## Step 1: Clean + select features (categorical columns kept as-is for now)

In [2]:
clean_missing_target(RAW_LISTINGS_PATH, CLEANED_TARGET_PATH, target_column=TARGET_COLUMN)
df = pd.read_csv(CLEANED_TARGET_PATH)

df = clean_numerical_features(df)
df = select_important_features(df)
df = clean_price_column(df)
df = remove_price_outliers_dynamic(df)
df = add_amenity_count(df)
df = add_description_word_count(df)

df[["room_type", "property_type", "neighbourhood_cleansed"]].nunique()


Reading /Users/jnanadeep/Desktop/airbnb-price-prediction/data/raw/listings.csv...



--- Processing Summary ---
Initial row count:          40769
Rows with missing 'price': 953 (Removed)
Remaining row count:        39816
Cleaned data saved to:      /Users/jnanadeep/Desktop/airbnb-price-prediction/data/processed/airbnb_rio_cleaned_target.csv



Dropping columns with >90.0% missing values:
 - neighborhood_overview
 - host_since
 - host_response_time
 - host_response_rate
 - host_acceptance_rate
 - host_thumbnail_url
 - host_neighbourhood
 - host_total_listings_count
 - host_verifications
 - neighbourhood
 - neighbourhood_group_cleansed
 - calendar_updated
 - license
 - instant_bookable

Imputed 25 numerical columns using their median value.
--- Running Feature Selection ---
Original unique columns: 76
Filtered columns retained: 24
--- Running Dynamic Outlier Removal ---
Calculated 99.0th percentile threshold: $6,999.13
Removed 399 listing(s) outside the range [$10 - $6,999.13].
New maximum price in dataset: $6,990.00


room_type                   4
property_type              81
neighbourhood_cleansed    155
dtype: int64

## Step 2: Single train/test split shared by both variants

In [3]:
y_log = np.log1p(df[TARGET_COLUMN])
train_df, test_df, y_train_log, y_test_log = train_test_split(
    df, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
)


## Step 3a: Baseline -- numeric-only features (current pipeline behavior)

In [4]:
numeric_cols = df.drop(columns=[TARGET_COLUMN, "id"], errors="ignore").select_dtypes(include=[np.number]).columns

X_train_baseline = train_df[numeric_cols]
X_test_baseline = test_df[numeric_cols]

baseline_model = train_xgboost(X_train_baseline, y_train_log)
baseline_preds_log = baseline_model.predict(X_test_baseline)
baseline_metrics = print_metrics_log_target("Baseline (numeric only)", y_test_log, baseline_preds_log)
baseline_metrics["n_features"] = X_train_baseline.shape[1]


Baseline (numeric only)        | R^2 Score: 61.74% | MAE: $255.68 | RMSE: $513.85


## Step 3b: + encoded categoricals

In [5]:
encoders = fit_categorical_encoders(train_df)

train_cat_features = transform_categorical_features(train_df, encoders)
test_cat_features = transform_categorical_features(test_df, encoders)

X_train_cat = pd.concat([X_train_baseline, train_cat_features], axis=1)
X_test_cat = pd.concat([X_test_baseline, test_cat_features], axis=1)

cat_model = train_xgboost(X_train_cat, y_train_log)
cat_preds_log = cat_model.predict(X_test_cat)
cat_metrics = print_metrics_log_target("+ categoricals", y_test_log, cat_preds_log)
cat_metrics["n_features"] = X_train_cat.shape[1]


+ categoricals                 | R^2 Score: 65.49% | MAE: $248.21 | RMSE: $501.43


## Step 4: Summary

In [6]:
comparison = pd.DataFrame(
    {"Baseline (numeric only)": baseline_metrics, "+ categoricals": cat_metrics}
).T[["n_features", "mae", "rmse", "r2"]]
comparison


,n_features,mae,rmse,r2
Baseline (numeric only),16.0,255.680693,513.852779,0.617386
+ categoricals,37.0,248.207831,501.426499,0.654851


## Step 5: Which categorical features mattered most?

In [7]:
import pandas as pd

importances = pd.Series(cat_model.feature_importances_, index=X_train_cat.columns)
new_cols = [c for c in X_train_cat.columns if c not in numeric_cols]
importances[new_cols].sort_values(ascending=False).head(15)


room_type_Entire home/apt                          0.115262
room_type_Shared room                              0.067017
room_type_Private room                             0.042151
property_type_Room in hotel                        0.018449
neighbourhood_cleansed_frequency                   0.014216
property_type_Private room in bed and breakfast    0.008524
property_type_Private room in rental unit          0.007581
property_type_Entire home                          0.006637
property_type_Shared room in rental unit           0.006133
property_type_Other                                0.006008
property_type_Private room in home                 0.005854
property_type_Room in aparthotel                   0.005049
property_type_Entire rental unit                   0.005044
property_type_Entire guesthouse                    0.004896
property_type_Entire serviced apartment            0.004737
dtype: float32